In [1]:
from matplotlib import pyplot as plt
from PIL import Image
import numpy as np
import os
from os import listdir

In [2]:

class ConvLayer:

    def __init__(self, num_filters=1, filter_size=3, input_channels=1):

        self.num_filters = num_filters
        self.filter_size = filter_size
        self.input_channels = input_channels

        self.filters = (
            np.random.randn(
                num_filters,
                input_channels,
                filter_size,
                filter_size
            ) * 0.1
        )

    def forward(self, image):

        # Convert a grayscale image into one input channel
        if image.ndim == 2:
            image = np.expand_dims(image, axis=0)

        self.image = image

        channels, h, w = image.shape
        fs = self.filter_size

        output = np.zeros(
            (self.num_filters, h-fs+1, w-fs+1)
        )

        for f in range(self.num_filters):
            for i in range(h-fs+1):
                for j in range(w-fs+1):

                    region = image[:, i:i+fs, j:j+fs]

                    output[f, i, j] = np.sum(
                        region * self.filters[f]
                    )

        return output

    def backward(self, d_out, learning_rate):

        fs = self.filter_size

        channels, h, w = self.image.shape

        d_filters = np.zeros_like(self.filters)
        d_input = np.zeros_like(self.image)

        for f in range(self.num_filters):
            for i in range(h-fs+1):
                for j in range(w-fs+1):

                    region = self.image[:, i:i+fs, j:j+fs]

                    d_filters[f] += d_out[f, i, j] * region

                    # Propagate the gradient to the previous layer
                    d_input[:, i:i+fs, j:j+fs] += (
                        d_out[f, i, j] * self.filters[f]
                    )

        self.filters -= learning_rate * d_filters

        return d_input


In [3]:

class MaxPool:

    def forward(self, x):

        self.input = x

        channels, h, w = x.shape

        output = np.zeros(
            (channels, h//2, w//2)
        )

        for c in range(channels):
            for i in range(0, h-1, 2):
                for j in range(0, w-1, 2):

                    region = x[c, i:i+2, j:j+2]

                    output[c, i//2, j//2] = np.max(region)

        return output

    def backward(self, grad):

        channels, h, w = self.input.shape

        d_input = np.zeros_like(self.input)

        for c in range(channels):
            for i in range(0, h-1, 2):
                for j in range(0, w-1, 2):

                    region = self.input[c, i:i+2, j:j+2]

                    max_pos = np.unravel_index(
                        np.argmax(region),
                        region.shape
                    )

                    d_input[
                        c,
                        i + max_pos[0],
                        j + max_pos[1]
                    ] = grad[c, i//2, j//2]

        return d_input


In [4]:
class Flatten:

    def forward(self, x):
        self.shape = x.shape
        return x.reshape(1, -1)

    def backward(self, grad):
        return grad.reshape(self.shape)

In [5]:
class ReLU:

    def forward(self, x):

        self.input = x

        return np.maximum(0, x)

    def backward(self, grad):

        return grad * (self.input > 0)

In [6]:
class Dense:

    def __init__(self, input_size, output_size):

        self.weights = (
            np.random.randn(input_size, output_size)
            * np.sqrt(2/input_size)
        )

        self.biases = np.zeros((1, output_size))

    def forward(self, x):

        self.input = x

        return np.dot(x, self.weights) + self.biases

    def backward(self, grad, lr):

        dw = np.dot(self.input.T, grad)

        db = np.sum(grad, axis=0, keepdims=True)

        dx = np.dot(grad, self.weights.T)

        self.weights -= lr * dw
        self.biases -= lr * db

        return dx

In [7]:
class Sigmoid:

    def forward(self, x):

        self.output = 1/(1+np.exp(-x))

        return self.output

    def backward(self, grad):

        return (
            grad *
            self.output *
            (1-self.output)
        )

In [8]:

conv1 = ConvLayer(num_filters=8, input_channels=1)
conv2 = ConvLayer(num_filters=16, input_channels=8)
conv3 = ConvLayer(num_filters=32, input_channels=16)

relu1 = ReLU()
relu2 = ReLU()
relu3 = ReLU()
relu4 = ReLU()

pool1 = MaxPool()
pool2 = MaxPool()

flat = Flatten()

dense1 = Dense(32 * 12 * 12, 64)
dense2 = Dense(64, 1)

sigmoid = Sigmoid()


In [9]:

def forward(image):

    # Block 1
    x = conv1.forward(image)
    x = relu1.forward(x)
    x = pool1.forward(x)

    # Block 2
    x = conv2.forward(x)
    x = relu2.forward(x)
    x = pool2.forward(x)

    # Block 3
    x = conv3.forward(x)
    x = relu3.forward(x)

    # Classifier
    x = flat.forward(x)

    x = dense1.forward(x)
    x = relu4.forward(x)

    x = dense2.forward(x)
    x = sigmoid.forward(x)

    return x


In [10]:
"""


"""
def loss(y_true, y_pred):

    eps = 1e-7

    return -(
        y_true*np.log(y_pred+eps)
        +
        (1-y_true)*np.log(
            1-y_pred+eps
        )
    )

In [ ]:


learning_rate = 0.01

folder_dir = "Big_Dataset/not_robin"

training_data = []
for images in os.listdir(folder_dir):
    if (images.endswith(".jpg")):
        img = Image.open(folder_dir+"/"+images)

        img = img.convert("L")

        img = img.resize((64,64))

        image = np.array(img)/255.0
        training_data.append((image, 0))

folder_dir = "Big_Dataset/robin"

for images in os.listdir(folder_dir):
    if (images.endswith(".jpg")):
        img = Image.open(folder_dir+"/"+images)

        img = img.convert("L")

        img = img.resize((64,64))

        image = np.array(img)/255.0
        training_data.append((image, 1))
        
epochs = 10

for epoch in range(epochs):

    total_loss = 0.0
    correct = 0

    np.random.shuffle(training_data)

    for image, label in training_data:

        y_true = np.array([[label]], dtype=float)

        # Forward pass
        prediction = forward(image)

        total_loss += loss(y_true, prediction)[0][0]

        predicted_class = int(prediction[0, 0] >= 0.5)

        if predicted_class == label:
            correct += 1


        grad = prediction - y_true

        grad = dense2.backward(grad, learning_rate)

        grad = relu4.backward(grad)

        grad = dense1.backward(grad, learning_rate)

        grad = flat.backward(grad)

    
        grad = relu3.backward(grad)
        grad = conv3.backward(grad, learning_rate)

        grad = pool2.backward(grad)
        grad = relu2.backward(grad)
        grad = conv2.backward(grad, learning_rate)

        grad = pool1.backward(grad)
        grad = relu1.backward(grad)
        grad = conv1.backward(grad, learning_rate)

    avg_loss = total_loss / len(training_data)
    accuracy = correct / len(training_data)


    print(
        f"Epoch {epoch + 1:3d}/{epochs} "
        f"Loss={avg_loss:.4f} "
        f"Accuracy={accuracy:.2%}"
    )

